# **Libraries**

In [ ]:
# Data Handling and Processing
import pandas as pd
import numpy as np
# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D  # for 3D plots
# Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
# Clustering Evaluation Metrics
from sklearn.metrics import davies_bouldin_score,silhouette_score, calinski_harabasz_score
# Clustering Models
from sklearn_extra.cluster import KMedoids
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import AgglomerativeClustering
# Dimensionality Reduction
from sklearn.decomposition import PCA  # Import PCA (Principal Component Analysis) for dimensionality reduction
# Classification Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
# Classification Evaluation Metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# **Cleaning on training data**

In [ ]:
# Keep only fields used in this exploratory analysis; do not load address/description fields.
from pathlib import Path
import os

DATA_PATH = Path(os.environ.get("CRIME_DATA_PATH", "train.csv")).expanduser()
if not DATA_PATH.is_absolute():
    DATA_PATH = Path.cwd() / DATA_PATH
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Crime data not found at {DATA_PATH}. Put an authorized train.csv in the "
        "working directory or set CRIME_DATA_PATH to an authorized local CSV."
    )

REQUIRED_COLUMNS = ["Dates", "Category", "DayOfWeek", "PdDistrict", "X", "Y"]
try:
    available_columns = pd.read_csv(DATA_PATH, nrows=0).columns
except (OSError, UnicodeError, pd.errors.ParserError) as exc:
    raise ValueError(f"Could not read the CSV header at {DATA_PATH}.") from exc
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(available_columns))
if missing_columns:
    raise ValueError(f"Input CSV is missing required columns: {missing_columns}")

df = pd.read_csv(DATA_PATH, usecols=REQUIRED_COLUMNS)
if df.empty:
    raise ValueError("Input CSV contains no incident rows.")
df["Dates"] = pd.to_datetime(df["Dates"], errors="coerce", format="mixed")
for column in ("X", "Y"):
    df[column] = pd.to_numeric(df[column], errors="coerce")
invalid_counts = df[REQUIRED_COLUMNS].isna().sum()
invalid_counts = invalid_counts[invalid_counts > 0]
if not invalid_counts.empty:
    details = ", ".join(f"{column}={int(count)}" for column, count in invalid_counts.items())
    raise ValueError(f"Input has missing or invalid required values ({details}); clean it locally before analysis.")

# The minimal validated columns are the only input fields retained below.
df.shape


In [ ]:
df.info()

In [ ]:
# check for missing data
df.isna().sum()

In [ ]:
# check for duplicates
df.duplicated().sum()

In [ ]:
# remove duplicates
df.drop_duplicates(inplace = True)

In [ ]:
# convert the date column form "object" to "datetime"
df['Dates'] = pd.to_datetime(df['Dates'])

In [ ]:
# display summary statistics
df.describe()

In [ ]:
plt.figure(figsize=(14, 6))
plt.subplot(2, 2, 1)
sns.boxplot(x=df['X'])
plt.title('Train - X (Longitude)')

plt.subplot(2, 2, 3)
sns.boxplot(x=df['Y'])
plt.title('Train - Y (Latitude)')
plt.tight_layout()

plt.show()

In [ ]:
Q1_X = df['X'].quantile(0.25)
Q3_X = df['X'].quantile(0.75)
IQR_X = Q3_X - Q1_X

Q1_Y = df['Y'].quantile(0.25)
Q3_Y = df['Y'].quantile(0.75)
IQR_Y = Q3_Y - Q1_Y

df = df[(df['X'] >= Q1_X - 1.5 * IQR_X) & (df['X'] <= Q3_X + 1.5 * IQR_X)]

df = df[(df['Y'] >= Q1_Y - 1.5 * IQR_Y) & (df['Y'] <= Q3_Y + 1.5 * IQR_Y)]


In [ ]:
# count the number of distinct categories
df['Category'].nunique()

In [ ]:
# Extract useful time-based features
df['Hour'] = df['Dates'].dt.hour
df['Month'] = df['Dates'].dt.month
df['Year'] = df['Dates'].dt.year
df['day'] = df['Dates'].dt.day
df['n_days'] = (df['Dates'] - df['Dates'].min()).apply(lambda x: x.days)

In [ ]:
# Encode categorical columns
le_category = LabelEncoder()
df['Category_encoded'] = le_category.fit_transform(df['Category'])

le_day = LabelEncoder()
df['DayOfWeek_encoded'] = le_day.fit_transform(df['DayOfWeek'])

le_district = LabelEncoder()
df['PdDistrict_encoded'] = le_district.fit_transform(df['PdDistrict'])

In [ ]:
df

In [ ]:
data = pd.DataFrame()
data[['X', 'Y','day','Month','Hour','n_days','PdDistrict_encoded','Category_encoded']] = df[['X', 'Y','day','Month','Hour','n_days','PdDistrict_encoded','Category_encoded']]
data.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

cols_to_scale = ['X', 'Y', 'day', 'Month', 'Hour', 'n_days']

data[cols_to_scale] = scaler.fit_transform(data[cols_to_scale])
df.head()

# **Visualization**

In [ ]:
# Crime distribution by category
sns.set_theme(style="whitegrid")
top_crimes = df['Category'].value_counts().head(15)
top_crimes_df = pd.DataFrame({'Category': top_crimes.index, 'Count': top_crimes.values})
plt.figure(figsize=(14, 9))
barplot = sns.barplot(data=top_crimes_df,x='Count',y='Category',hue='Category', palette='coolwarm',dodge=False,legend=False)
plt.title('Top 15 Crime Categories in San Francisco', fontsize=16, weight='bold')
plt.xlabel('Number of Incidents', fontsize=12)
plt.ylabel('Crime Category', fontsize=12)
for i, value in enumerate(top_crimes_df['Count']):
    plt.text(value + 100, i, f'{value:,}', va='center', fontsize=10)
plt.xticks(fontsize=10)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.savefig('top_crimes.png')
plt.show()

In [ ]:
# Crimes by district
plt.figure(figsize=(10, 6))
district_counts = df['PdDistrict'].value_counts()
colors = plt.cm.Blues(np.linspace(0.3, 1, len(district_counts)))
plt.pie(district_counts, labels=district_counts.index, autopct='%1.1f%%', colors=colors)
plt.title('Crime Distribution by District', fontsize=14, weight='bold')
plt.axis('equal')
plt.tight_layout()
plt.savefig('crimes_by_district.png')
plt.show()

In [ ]:
# Crimes by time of day
plt.figure(figsize=(12, 6))
crime_by_hour = df.groupby('Hour')['Category'].count()
plt.plot(crime_by_hour.index + 1, crime_by_hour.values,marker='o',linestyle='-',linewidth=2,color='#483D8B',markerfacecolor='#9370DB',markeredgecolor='white')
plt.title('Crime Incidents by Hour of Day', fontsize=16, weight='bold')
plt.xlabel('Hour (24-hour format)', fontsize=12)
plt.ylabel('Number of Incidents', fontsize=12)
plt.xticks(range(1, 25), fontsize=10)
plt.yticks(fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('crimes_by_hour.png')
plt.show()

# **Clustering Models**

In [ ]:
# Correlation Heatmap of Features
corr_matrix =data.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap='PuBu',
    linewidths=0.5,
    cbar_kws={'label': 'Correlation Coefficient'}
)
plt.title('Correlation Heatmap of Features', fontsize=16, weight='bold')
plt.tight_layout()
plt.savefig('Correlation Heatmap.png')
plt.show()

In [ ]:
# unbiased sample
sampled_data, _ = train_test_split(
    data,
    train_size=25000,
    stratify=data['Category_encoded'],
    random_state=42
)

# select feature columns
X = sampled_data.drop(columns=['Category_encoded'])

In [ ]:
X.head()

In [ ]:
range_n_clusters = range(2, 15)
silhouette_scores = []

for k in range_n_clusters:
    kmedoids = KMedoids(n_clusters=k, random_state=0)
    kmedoids.fit(X)

    score = silhouette_score(X, kmedoids.labels_)
    silhouette_scores.append(score)

plt.figure(figsize=(10, 6))
plt.plot(range_n_clusters, silhouette_scores, marker='o', color='green')
plt.title("Silhouette Score vs Number of Clusters (KMedoids)")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score (Higher is Better)")
plt.grid(True)
plt.show()
#select best k
best_k = range_n_clusters[silhouette_scores.index(max(silhouette_scores))]

print(f"Best k found: {best_k}")

In [ ]:
# apply kmedoids using the best k value
kmedoids = KMedoids(n_clusters=best_k, random_state=0).fit(X)
# create a new dataframe containing the labels of the cluster
clustered_df = X.copy()
clustered_df['New_Labels'] = kmedoids.labels_

# Evaluation metrics for the train dataset:
features = clustered_df.drop(columns=['New_Labels'])

# 1- silhouette_score better to be big
sil_score_train = silhouette_score(features, kmedoids.labels_)
# 2- calinski_harabasz_score better to be big
calinski_score_train = calinski_harabasz_score(features, kmedoids.labels_)
# 3- davies_bouldin_score better to be small
davies_score_train = davies_bouldin_score(features, kmedoids.labels_)

print("Silhouette Score (Clustering Sample):", sil_score_train)
print("Calinski-Harabasz Index (Clustering Sample):", calinski_score_train)
print("Davies-Bouldin Index (Clustering Sample):", davies_score_train)
print("****************************************")

In [ ]:
clustered_df

In [ ]:
# Correlation Heatmap of Features of the NEW clustered dataset
corr_matrix =clustered_df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap='PuBu',
    linewidths=0.5,
    cbar_kws={'label': 'Correlation Coefficient'}
)
plt.title('Correlation Heatmap of Features', fontsize=16, weight='bold')
plt.tight_layout()
plt.savefig('Correlation Heatmap.png')
plt.show()

In [ ]:
# Create 2D plot using any two columns from X (assuming X is a pandas DataFrame)
plt.figure(figsize=(10, 6))

# Scatter plot with cluster labels as colors
sc = plt.scatter(X.iloc[:, 0], X.iloc[:, 1], c=kmedoids.labels_, cmap='tab10', s=15)

plt.title("KMedoids Clustering Visualization (2D Scatter)", fontsize=16)
plt.xlabel("Feature 1")  # Replace with your feature names if needed
plt.ylabel("Feature 2")  # Replace with your feature names if needed

plt.colorbar(sc, label='Cluster')
plt.show()


In [ ]:
# Apply PCA (Principal Component Analysis) to reduce data to 3D
pca = PCA(n_components=3) # choose 3D
reduced_data_3d = pca.fit_transform(X) # Transform the original data (X) to 3D

# Create 3D plot
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot with cluster labels as colors
sc = ax.scatter(reduced_data_3d[:, 0], reduced_data_3d[:, 1], reduced_data_3d[:, 2],
                c=kmedoids.labels_, cmap='tab10', s=15)

ax.set_title("KMedoids Clustering Visualization (PCA 3D)", fontsize=16)
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
ax.set_zlabel("PCA 3")

plt.colorbar(sc, label='Cluster')
plt.show()

* Dendogram

In [ ]:
plt.figure(figsize=(16, 6))
linked=linkage(X,metric='euclidean', method='average')
dendrogram(linked, truncate_mode='lastp', p=30, leaf_rotation=90., leaf_font_size=10., show_contracted=True)
plt.axhline(y=4.65, color='red', linestyle='--') 
plt.title("Hierarchical Clustering Dendrogram (complete)")
plt.xlabel("Data Points or Clusters")
plt.ylabel("Distance")
plt.tight_layout()
plt.savefig('hierarchical_dendrogram_cut.png')
plt.show()


In [ ]:
hierarchical_model = AgglomerativeClustering(n_clusters=7, metric='euclidean', linkage='average')
hierarchical_labels = hierarchical_model.fit_predict(X)
hierar_df=X.copy()
hierar_df['Hierarchical_Cluster'] = hierarchical_labels
hierar_df.head()

In [ ]:
# Evaluation metrics
# get the feature columns without the clutser column
features = hierar_df.drop(columns=['Hierarchical_Cluster'])
# evaluate on the feature columns
sil_score = silhouette_score(features, hierarchical_labels)
calinski_score = calinski_harabasz_score(features, hierarchical_labels)
davies_score = davies_bouldin_score(features, hierarchical_labels)

print("Silhouette Score :", sil_score)
print("Calinski-Harabasz Index :", calinski_score)
print("Davies-Bouldin Index :", davies_score)

In [ ]:
results = {
    "Metric": ["Silhouette Score", "Calinski-Harabasz Index", "Davies-Bouldin Index"],
    "KMedoids": [sil_score_train, calinski_score_train, davies_score_train],
    "Hierarchical": [sil_score, calinski_score, davies_score]
}

compare_df = pd.DataFrame(results)
compare_df

In [ ]:
better_k = int(sil_score_train > sil_score) + int(calinski_score_train > calinski_score) + int(davies_score_train < davies_score)
better_h = 3 - better_k
if better_k > better_h:
    print("KMedoids is the better algorithm based on the metrics.")
elif better_h > better_k:
    print("Hierarchical is the better algorithm based on the metrics.")
else:
    print("Both algorithms performed equally based on the metrics.")

# **Classification Models**

> The target is a K-Medoids-generated cluster label, not an observed crime category or future crime outcome. Preprocessing, feature scaling, and clustering are performed before this row-level split, using the full analysis sample; therefore validation/test labels are not independent and this is not a leakage-free generalization estimate. Classifier scores measure imitation of labels generated from the same sample only. The random split is not time- or location-held-out and must not be used to claim crime-prediction performance or support decisions about people or places.


In [ ]:
# select the feature columns, and the target (new_labels that came from kmedoids)
X_new = clustered_df.drop(columns=['New_Labels'])
y = clustered_df['New_Labels']

In [ ]:
from sklearn.model_selection import train_test_split
# Row-wise, stratified 60/20/20 split. This improves label balance but does NOT undo
# the full-sample preprocessing/clustering leakage described above.
x_train, x_temp, y_train, y_temp = train_test_split(
    X_new, y, test_size=0.4, random_state=43, stratify=y
)
# Split the temporary 40% evenly: 20% validation and 20% test overall.
x_test, x_val, y_test, y_val = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=43, stratify=y_temp
)
assert len(x_train) + len(x_val) + len(x_test) == len(X_new)
assert set(x_train.index).isdisjoint(x_val.index)
assert set(x_train.index).isdisjoint(x_test.index)
assert set(x_val.index).isdisjoint(x_test.index)


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Train the model on training data
tree_model = DecisionTreeClassifier(max_depth=15, min_samples_split=100, min_samples_leaf=50, criterion='entropy')
tree_model.fit(x_train, y_train)

# 2. Evaluate on Train data
y_pred_tree_train = tree_model.predict(x_train)
print("Decision Tree (Train) Results:")
print("Accuracy of Decision Tree (Train):", accuracy_score(y_train, y_pred_tree_train))
print(classification_report(y_train, y_pred_tree_train, zero_division=0))
print("****************************************")

# 3. Evaluate on Validation data
y_pred_tree_val = tree_model.predict(x_val)
print("Decision Tree (Validation) Results:")
print("Accuracy of Decision Tree (Validation):", accuracy_score(y_val, y_pred_tree_val))
print(classification_report(y_val, y_pred_tree_val, zero_division=0))
print("****************************************")

# 4. Evaluate on Test data
y_pred_tree_test = tree_model.predict(x_test)
print("Decision Tree (Test) Results:")
print("Accuracy of Decision Tree (Test):", accuracy_score(y_test, y_pred_tree_test))
print(classification_report(y_test, y_pred_tree_test, zero_division=0))


In [ ]:
cm_tree_test = confusion_matrix(y_test, y_pred_tree_test)
labels = [f"Cluster {i}" for i in range(cm_tree_test.shape[0])]
y_pred_tree_test = tree_model.predict(x_test)
plt.figure(figsize=(7,5))
sns.heatmap(
    cm_tree_test,
    annot=True,
    fmt="d",
    cmap="Oranges_r",
    cbar=False,
    xticklabels=labels,
    yticklabels=labels,
    linewidths=0.5,
    linecolor='gray'
)
plt.title("Confusion Matrix - Decision Tree (Test Data)", fontsize=14, weight='bold')
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
# Initialize RandomForestClassifier
rf_best = RandomForestClassifier(
    bootstrap=True,
    max_depth=70,
    max_features='sqrt',
    min_samples_leaf=4,
    min_samples_split=10,
    n_estimators=400,
    random_state=42
)

# Train the model on train data
rf_best.fit(x_train, y_train)
# Predict on train data
y_pred_rf_train = rf_best.predict(x_train)
print("RandomForest (Train) Results:")
print("Accuracy of RandomForest (Train):", accuracy_score(y_train, y_pred_rf_train))
print("Classification Report (Train):\n", classification_report(y_train, y_pred_rf_train, zero_division=0))
print("****************************************")

# Evaluate on Validation data
y_pred_rf_val = rf_best.predict(x_val)
print("RandomForest (Validation) Results:")
print("Accuracy of RandomForest (Validation):", accuracy_score(y_val, y_pred_rf_val))
print("Classification Report (Validation):\n", classification_report(y_val, y_pred_rf_val, zero_division=0))
print("****************************************")

# Evaluate on Test data
y_pred_rf_test = rf_best.predict(x_test)
print("RandomForest (Test) Results:")
print("Accuracy of RandomForest (Test):", accuracy_score(y_test, y_pred_rf_test))
print("Classification Report (Test):\n", classification_report(y_test, y_pred_rf_test, zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, y_pred_rf_test)
labels = [f"Cluster {i}" for i in range(cm.shape[0])]
plt.figure(figsize=(10, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=labels,
    yticklabels=labels,
    linewidths=0.5,
    linecolor='gray'
)
plt.title("Confusion Matrix - Random Forest (Test Data)", fontsize=14, weight='bold')
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()